# Práctica 5 · Recomendador de películas

## 1. Filtro colaborativo con MovieLens

Se usan las valoraciones de MovieLens Latest Small. SVD representa a cada usuario y a cada película con 20 factores y los usa para estimar valoraciones.

In [1]:
from pathlib import Path
from zipfile import ZipFile

import pandas as pd
from IPython.display import display
from surprise import Dataset, Reader, SVD

SEMILLA = 42
COLUMNAS = ["userId", "movieId", "rating"]
ESCALA = (0.5, 5)
ruta_zip = Path("ml-latest-small.zip")
if not ruta_zip.is_file():
    ruta_zip = Path(".data") / ruta_zip

# Lee las valoraciones y el catalogo
with ZipFile(ruta_zip) as archivo:
    valoraciones = pd.read_csv(archivo.open("ml-latest-small/ratings.csv"))
    peliculas = pd.read_csv(archivo.open("ml-latest-small/movies.csv"))

print("Valoraciones:", len(valoraciones))
print("Usuarios:", valoraciones["userId"].nunique())
print("Películas del catálogo:", len(peliculas))

Valoraciones: 100836
Usuarios: 610
Películas del catálogo: 9742


In [2]:
lector = Reader(rating_scale=ESCALA)
entrenamiento = Dataset.load_from_df(valoraciones[COLUMNAS], lector).build_full_trainset()
ajustes_svd = {"n_factors": 20, "random_state": SEMILLA}

# Entrena el filtro con los usuarios originales
modelo_base = SVD(**ajustes_svd).fit(entrenamiento)
print("Valoraciones usadas para entrenar SVD:", modelo_base.trainset.n_ratings)

Valoraciones usadas para entrenar SVD: 100836


## 2. Diez recomendaciones por perfil añadido

Se añaden dos perfiles con valoraciones de ejemplo. El perfil A suele puntuar mejor las películas de animación y el perfil B las de crimen. Se vuelve a entrenar SVD con ambos perfiles y se muestran sus diez recomendaciones entre las películas que todavía no han puntuado.

In [3]:
perfiles = {
    "Perfil A": {1: 5, 2: 4, 6: 2, 50: 3, 296: 2.5, 318: 4, 588: 5, 595: 5},
    "Perfil B": {1: 3, 2: 2.5, 6: 5, 50: 5, 296: 5, 318: 4.5, 588: 2, 595: 2.5},
}
primer_id = valoraciones["userId"].max() + 1
ids_perfiles = dict(zip(perfiles, range(primer_id, primer_id + len(perfiles))))
filas_nuevas = []
for nombre, notas in perfiles.items():
    for pelicula, nota in notas.items():
        filas_nuevas.append((ids_perfiles[nombre], pelicula, nota))

nuevas = pd.DataFrame(filas_nuevas, columns=COLUMNAS)
if not nuevas["movieId"].isin(valoraciones["movieId"]).all():
    raise ValueError("Los perfiles deben usar películas con valoraciones en MovieLens.")
if not nuevas["rating"].between(*ESCALA).all():
    raise ValueError("Las notas de los perfiles deben estar entre 0,5 y 5.")
display(nuevas.merge(peliculas[["movieId", "title"]], on="movieId"))

,userId,movieId,rating,title
0,611,1,5.0,Toy Story (1995)
1,611,2,4.0,Jumanji (1995)
2,611,6,2.0,Heat (1995)
3,611,50,3.0,"Usual Suspects, The (1995)"
4,611,296,2.5,Pulp Fiction (1994)
5,611,318,4.0,"Shawshank Redemption, The (1994)"
6,611,588,5.0,Aladdin (1992)
7,611,595,5.0,Beauty and the Beast (1991)
8,612,1,3.0,Toy Story (1995)
9,612,2,2.5,Jumanji (1995)


In [4]:
completo = pd.concat([valoraciones[COLUMNAS], nuevas], ignore_index=True)
datos_completos = Dataset.load_from_df(completo, lector).build_full_trainset()
modelo_final = SVD(**ajustes_svd).fit(datos_completos)

ids_valorados = valoraciones["movieId"].unique()
catalogo = peliculas[peliculas["movieId"].isin(ids_valorados)]
for nombre, usuario in ids_perfiles.items():
    # Quita las peliculas ya puntuadas por el perfil
    candidatas = catalogo.loc[~catalogo["movieId"].isin(perfiles[nombre])].copy()
    candidatas["Estimación"] = [
        modelo_final.predict(usuario, pelicula, clip=False).est
        for pelicula in candidatas["movieId"]
    ]
    lista = candidatas.nlargest(10, "Estimación")[["title", "genres", "Estimación"]].copy()
    lista["Estimación"] = lista["Estimación"].clip(*ESCALA).round(3)
    print(nombre)
    display(lista.reset_index(drop=True))

Perfil A


,title,genres,Estimación
0,"Streetcar Named Desire, A (1951)",Drama,4.376
1,Amadeus (1984),Drama,4.297
2,Guess Who's Coming to Dinner (1967),Drama,4.244
3,"Princess Bride, The (1987)",Action|Adventure|Comedy|Fantasy|Romance,4.234
4,Raging Bull (1980),Drama,4.227
5,"Philadelphia Story, The (1940)",Comedy|Drama|Romance,4.204
6,American History X (1998),Crime|Drama,4.199
7,Glory (1989),Drama|War,4.191
8,"Grand Day Out with Wallace and Gromit, A (1989)",Adventure|Animation|Children|Comedy|Sci-Fi,4.190
9,In the Name of the Father (1993),Drama,4.190


Perfil B


,title,genres,Estimación
0,Fight Club (1999),Action|Crime|Drama|Thriller,4.256
1,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War,4.252
2,Rear Window (1954),Mystery|Thriller,4.217
3,Lawrence of Arabia (1962),Adventure|Drama|War,4.210
4,"Streetcar Named Desire, A (1951)",Drama,4.179
5,12 Angry Men (1957),Drama,4.177
6,Blade Runner (1982),Action|Sci-Fi|Thriller,4.170
7,Life Is Beautiful (La Vita è bella) (1997),Comedy|Drama|Romance|War,4.170
8,Ghost in the Shell (Kôkaku kidôtai) (1995),Animation|Sci-Fi,4.169
9,There Will Be Blood (2007),Drama|Western,4.166


## 3. Sesgos de usuarios y películas más valoradas

Un sesgo negativo indica que el modelo tiende a estimar notas más bajas para ese usuario. Uno positivo indica lo contrario. La tabla muestra los cinco usuarios originales de cada extremo.

In [5]:
# Relaciona el orden interno de Surprise con los IDs de MovieLens
usuarios_modelo = [datos_completos.to_raw_uid(i) for i in datos_completos.all_users()]
sesgo_usuario = pd.Series(modelo_final.bu, index=usuarios_modelo, name="Sesgo")
resumen_usuarios = valoraciones.groupby("userId")["rating"].agg(Notas="size", Media="mean")
resumen_usuarios = resumen_usuarios.join(sesgo_usuario)
extremos_usuarios = pd.concat([
    resumen_usuarios.nsmallest(5, "Sesgo"),
    resumen_usuarios.nlargest(5, "Sesgo"),
])
display(extremos_usuarios.round(3))

,Notas,Media,Sesgo
userId,,,
442,20,1.275,-1.916
153,179,2.218,-1.430
139,194,2.144,-1.427
567,385,2.245,-1.377
508,24,2.146,-1.357
53,20,5.000,1.233
43,114,4.553,1.160
452,202,4.557,0.947
12,32,4.391,0.927


Las películas se ordenan por el número de veces que se valoraron en MovieLens. La media y el sesgo estimado indican cómo se puntuó cada título.

In [6]:
peliculas_modelo = [datos_completos.to_raw_iid(i) for i in datos_completos.all_items()]
sesgo_pelicula = pd.Series(modelo_final.bi, index=peliculas_modelo, name="Sesgo")
resumen_peliculas = valoraciones.groupby("movieId")["rating"].agg(Notas="size", Media="mean")
resumen_peliculas = resumen_peliculas.join(sesgo_pelicula)
resumen_peliculas = resumen_peliculas.join(peliculas.set_index("movieId")["title"])

print("Películas que recibieron más notas")
populares = resumen_peliculas.nlargest(10, "Notas")
display(populares[["title", "Notas", "Media", "Sesgo"]].round(3))

print("Usuario con menor sesgo:", sesgo_usuario.loc[resumen_usuarios.index].idxmin())
print("Usuario con mayor sesgo:", sesgo_usuario.loc[resumen_usuarios.index].idxmax())
print("Película con más notas:", populares.iloc[0]["title"])

Películas que recibieron más notas


,title,Notas,Media,Sesgo
movieId,,,,
356,Forrest Gump (1994),329,4.164,0.695
318,"Shawshank Redemption, The (1994)",317,4.429,0.900
296,Pulp Fiction (1994),307,4.197,0.704
593,"Silence of the Lambs, The (1991)",279,4.161,0.662
2571,"Matrix, The (1999)",278,4.192,0.714
260,Star Wars: Episode IV - A New Hope (1977),251,4.231,0.732
480,Jurassic Park (1993),238,3.750,0.314
110,Braveheart (1995),237,4.032,0.568
589,Terminator 2: Judgment Day (1991),224,3.971,0.482


Usuario con menor sesgo: 442
Usuario con mayor sesgo: 53
Película con más notas: Forrest Gump (1994)


## 4. Diez películas cercanas a Toy Story

La similitud coseno compara la dirección de los vectores de dos películas. Cuanto mayor es el valor, más cerca están en el espacio de factores del modelo. Se usa el identificador 1 de MovieLens, correspondiente a Toy Story (1995). Los vectores se aprenden de las valoraciones. Por eso aparecen películas de otros géneros. El número de notas ayuda a interpretar la cercanía cuando el título tiene poco historial.

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

consulta = 1
fila_consulta = datos_completos.to_inner_iid(consulta)
similitudes = cosine_similarity(modelo_final.qi[[fila_consulta]], modelo_final.qi)[0]
vecinas = pd.DataFrame({"movieId": peliculas_modelo, "Coseno": similitudes})
# Elimina Toy Story de sus propias vecinas
vecinas = vecinas[vecinas["movieId"] != consulta].nlargest(10, "Coseno")
vecinas = vecinas.merge(catalogo, on="movieId")
vecinas = vecinas.join(resumen_peliculas["Notas"], on="movieId")
display(vecinas[["title", "genres", "Notas", "Coseno"]].round(3))

,title,genres,Notas,Coseno
0,9 1/2 Weeks (Nine 1/2 Weeks) (1986),Drama|Romance,10,0.699
1,"Two Jakes, The (1990)",Drama,2,0.685
2,Not Another Teen Movie (2001),Comedy,14,0.683
3,Iron Man 3 (2013),Action|Sci-Fi|Thriller|IMAX,32,0.674
4,Act of Valor (2012),Action|Thriller|War,2,0.670
5,"Cage aux Folles, La (1978)",Comedy,1,0.650
6,Below (2002),Horror,1,0.649
7,National Velvet (1944),Children|Drama,2,0.648
8,"Magdalene Sisters, The (2002)",Drama,6,0.638
9,20 Dates (1998),Comedy|Romance,7,0.635


## 5. Grupos de usuarios

K-means agrupa a los usuarios originales según sus factores latentes. Antes de agruparlos, los vectores se normalizan para comparar su dirección. Se prueban de dos a seis grupos y se elige la silueta media más alta. Una silueta próxima a cero indica que los grupos se solapan. Para interpretar los grupos, se comparan la actividad, el sesgo y los géneros que aparecen con más frecuencia entre las notas de cuatro estrellas o más.

In [8]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import normalize
from threadpoolctl import threadpool_limits

filas = [datos_completos.to_inner_uid(u) for u in resumen_usuarios.index]
factores = normalize(modelo_final.pu[filas])
modelos_grupos = {}
siluetas = {}
# Compara varios valores de k sobre los mismos usuarios
with threadpool_limits(limits=1):
    for k in range(2, 7):
        modelo = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(factores)
        modelos_grupos[k] = modelo
        siluetas[k] = silhouette_score(factores, modelo.labels_)

mejor_k = max(siluetas, key=siluetas.get)
grupos = pd.Series(modelos_grupos[mejor_k].labels_ + 1, index=resumen_usuarios.index, name="Grupo")
display(pd.Series(siluetas, name="Silueta").rename_axis("k").round(3).to_frame())

,Silueta
k,
2,0.059
3,0.049
4,0.044
5,0.044
6,0.044


In [9]:
resumen = resumen_usuarios.join(grupos).groupby("Grupo").agg(
    Usuarios=("Notas", "size"),
    Mediana_notas=("Notas", "median"),
    Media_notas=("Media", "mean"),
    Sesgo_medio=("Sesgo", "mean"),
)
display(resumen.round(3))

,Usuarios,Mediana_notas,Media_notas,Sesgo_medio
Grupo,,,,
1,300,84.0,3.635,-0.042
2,310,64.0,3.678,0.027


Los porcentajes cuentan menciones de géneros entre las notas de cuatro estrellas o más. Una película puede aportar varias menciones.

In [10]:
altas = valoraciones[valoraciones["rating"] >= 4].join(grupos, on="userId")
altas = altas.merge(peliculas[["movieId", "genres"]], on="movieId")
altas["genero"] = altas["genres"].str.split("|", regex=False)
generos = altas.explode("genero")
generos = generos[generos["genero"] != "(no genres listed)"]
for grupo, notas in generos.groupby("Grupo"):
    # Una pelicula puede contar en varios generos
    favoritos = notas["genero"].value_counts(normalize=True).head(3)
    etiquetas = [f"{genero} ({proporcion:.1%})" for genero, proporcion in favoritos.items()]
    print(f"Grupo {grupo}: {', '.join(etiquetas)}")

print(f"Se eligieron {mejor_k} grupos con una silueta de {siluetas[mejor_k]:.3f}.")

Grupo 1: Drama (18.2%), Comedy (12.6%), Action (10.0%)
Grupo 2: Drama (15.9%), Comedy (12.7%), Action (10.9%)
Se eligieron 2 grupos con una silueta de 0.059.


En esta ejecución, los usuarios del primer grupo suelen valorar más películas. Los géneros más frecuentes se parecen en ambos grupos y la silueta baja indica que se solapan bastante. Los grupos sirven para describir diferencias de actividad dentro de MovieLens.